# Step 5 — Feature Engineering

## Goal

Create the numeric and text features required by the GameGuide
search, ranking, and recommendation engines.

The engineered features will include:

- Bayesian weighted rating
- Popularity score
- Number of supported languages
- Singleplayer / Multiplayer / Co-op indicators
- Recency score
- Normalized text fields
- Combined TF-IDF similarity text (`soup`)

The final engineered dataset will be saved as:

`data/processed/games_features.parquet`

In [2]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src import config

df = pd.read_parquet("../data/processed/games_clean.parquet")

print("Cleaned dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Cleaned dataset loaded successfully!
Shape: 89,550 rows × 53 columns


## 5.1 Derived Features

We verify the existing derived columns from Step 2 (`release_year`, `is_free`, `price_inr`), include `name_clean` from Step 3 for fast game-name lookup, and engineer the remaining features required by the engines:

- `release_year`: Year extracted from `release_date`
- `is_free`: Boolean (`price == 0`)
- `price_inr`: Price converted to Indian Rupees (`price * config.USD_TO_INR`)
- `name_clean`: Normalized game name (lowercase, punctuation stripped)
- `n_languages`: Count of `supported_languages`
- `is_singleplayer`, `is_multiplayer`, `is_coop`: Boolean flags derived from `categories`
- `recency`: Min-max normalized `release_year` in `[0, 1]` (newer is higher)
- `popularity_score`: `np.log1p(num_reviews_total)` blended with log-scaled `owners_mid` and `peak_ccu`

In [3]:
# 2. Count of supported languages (n_languages)
df["n_languages"] = df["supported_languages"].apply(
    lambda lst: len(lst) if isinstance(lst, (list, tuple, set, np.ndarray)) else 0
).astype("int32")

# 3. Gameplay mode flags derived from categories
def has_category_keyword(cats, keywords):
    if not isinstance(cats, (list, tuple, set, np.ndarray)):
        return False
    cats_lower = [str(c).lower() for c in cats]
    return any(any(kw in c for kw in keywords) for c in cats_lower)

df["is_singleplayer"] = df["categories"].apply(
    lambda c: has_category_keyword(c, ["single-player", "singleplayer"])
).astype(bool)

df["is_multiplayer"] = df["categories"].apply(
    lambda c: has_category_keyword(c, ["multi-player", "multiplayer", "pvp"])
).astype(bool)

df["is_coop"] = df["categories"].apply(
    lambda c: has_category_keyword(c, ["co-op", "coop"])
).astype(bool)

print(f"n_languages — min: {df['n_languages'].min()}, median: {df['n_languages'].median():.0f}, max: {df['n_languages'].max()}")
print(f"is_singleplayer: {df['is_singleplayer'].sum():,} ({df['is_singleplayer'].mean()*100:.2f}%)")
print(f"is_multiplayer:  {df['is_multiplayer'].sum():,} ({df['is_multiplayer'].mean()*100:.2f}%)")
print(f"is_coop:         {df['is_coop'].sum():,} ({df['is_coop'].mean()*100:.2f}%)")

n_languages — min: 0, median: 1, max: 103
is_singleplayer: 84,398 (94.25%)
is_multiplayer:  16,419 (18.34%)
is_coop:         8,795 (9.82%)


In [4]:
# 4. Recency: normalized release_year (newer is higher)
year_min = df["release_year"].min()
year_max = df["release_year"].max()
df["recency"] = ((df["release_year"] - year_min) / (year_max - year_min + 1e-9)).round(6)

# 5. Popularity score: log1p(num_reviews_total) blended with peak_ccu and owners_mid
v_reviews = df["num_reviews_total"].fillna(df["positive"] + df["negative"]).fillna(0).clip(lower=0)
log_reviews = np.log1p(v_reviews)
log_owners = np.log1p(df["owners_mid"].fillna(0).clip(lower=0))
log_ccu = np.log1p(df["peak_ccu"].fillna(0).clip(lower=0))

df["popularity_score"] = (0.75 * log_reviews + 0.15 * log_owners + 0.10 * log_ccu).round(6)

print(f"recency — min: {df['recency'].min():.4f}, median: {df['recency'].median():.4f}, max: {df['recency'].max():.4f}")
print(f"popularity_score — min: {df['popularity_score'].min():.4f}, median: {df['popularity_score'].median():.4f}, max: {df['popularity_score'].max():.4f}")

recency — min: 0.0000, median: 0.8571, max: 1.0000
popularity_score — min: 0.0000, median: 3.4822, max: 16.2031


### Popularity Score Design

The popularity score combines three popularity signals after logarithmic
transformation:

- Review volume: 75%
- Estimated owners: 15%
- Peak concurrent users: 10%

Log transformation reduces the effect of extreme popularity values while
preserving the relative ordering between games.

In [5]:
print("Sample derived features:")
print(df[["name", "release_year", "recency", "n_languages", "is_singleplayer", "is_multiplayer", "is_coop", "popularity_score"]].head(5).to_string())

Sample derived features:
                              name  release_year   recency  n_languages  is_singleplayer  is_multiplayer  is_coop  popularity_score
0                 Counter-Strike 2          2012  0.535714           29            False            True    False         16.203051
1              PUBG: BATTLEGROUNDS          2017  0.714286           17            False            True    False         15.106162
2                           Dota 2          2013  0.571429           28            False            True     True         15.308358
3        Grand Theft Auto V Legacy          2015  0.642857           13             True            True     True         14.691607
4  Tom Clancy's Rainbow Six® Siege          2015  0.642857           16             True            True     True         11.859929


## 5.2 Bayesian (Weighted) Rating (`rating_score`)

This prevents a game with 3 reviews at 100% positive from beating a game with 50,000 reviews at 95% positive:

$$\text{WR} = \left(\frac{v}{v + m}\right) R + \left(\frac{m}{v + m}\right) C$$

- $v$: reviews of the game (`num_reviews_total`)
- $R$: positive ratio of the game (`positive / (positive + negative)`)
- $m$: minimum-review threshold (70th percentile of `num_reviews_total`)
- $C$: mean positive ratio across all games

In [6]:
# Compute Bayesian (weighted) rating
R = df["positive"] / (df["positive"] + df["negative"]).replace(0, np.nan)
C = R.mean()
m = df["num_reviews_total"].quantile(0.70)
v = df["num_reviews_total"].fillna(0)

df["rating_score"] = ((v / (v + m)) * R.fillna(C) + (m / (v + m)) * C).round(6)

print(f"Mean positive ratio across all games (C): {C:.4f} ({C*100:.2f}%)")
print(f"70th percentile minimum-review threshold (m): {m:.1f}")
print(f"Missing values in rating_score: {df['rating_score'].isna().sum()}")
print("\nTop 10 Games by Bayesian rating_score:")
print(
    df.sort_values("rating_score", ascending=False)[
        ["name", "positive", "negative", "num_reviews_total", "pct_pos_total", "rating_score"]
    ].head(10).to_string()
)

Mean positive ratio across all games (C): 0.7540 (75.40%)
70th percentile minimum-review threshold (m): 165.0
Missing values in rating_score: 0

Top 10 Games by Bayesian rating_score:
                               name  positive  negative  num_reviews_total  pct_pos_total  rating_score
1042                   A Short Hike     18208       151            15537.0           99.0      0.989277
1450        Papa's Freezeria Deluxe     10682        78            10014.0           99.0      0.988881
663                    Senren＊Banka     29679       330            27223.0           98.0      0.987588
2075        TOEM: A Photo Adventure      7747        60             5894.0           99.0      0.985825
58                People Playground    271156      3874           258453.0           98.0      0.985766
1250  Our Life: Beginnings & Always     11965       142            12130.0           98.0      0.985128
67                Vampire Survivors    243277      3737           234756.0           98.

In [7]:
# Verify that acclaimed games rank well above 3-review 100% positive games
acclaimed = df[df["name"].isin(["Stardew Valley", "Portal 2", "Hades", "The Witcher 3: Wild Hunt", "ELDEN RING"])]
small_100 = df[(df["positive"] == 3) & (df["negative"] == 0)].head(2)

comp = pd.concat([acclaimed, small_100])[
    ["name", "positive", "negative", "num_reviews_total", "pct_pos_total", "rating_score"]
].sort_values("rating_score", ascending=False)

print("Comparison — Well-Known Acclaimed Games vs. 3-Review 100% Positive Games:")
print(comp.to_string())

Comparison — Well-Known Acclaimed Games vs. 3-Review 100% Positive Games:
                           name  positive  negative  num_reviews_total  pct_pos_total  rating_score
17               Stardew Valley    841413     13440           733626.0           98.0      0.984226
61                        Hades    271755      4795           255511.0           98.0      0.982514
15     The Witcher 3: Wild Hunt    791139     31979           751939.0           96.0      0.961104
16                   ELDEN RING    956144     73714           749606.0           92.0      0.928385
83210        Last Space Mission         3         0                NaN            NaN      0.754020
83211    ابو فانوس - Abu Fanous         3         0                NaN            NaN      0.754020


## 5.3 Text Field for Similarity (`soup`)

We construct the combined text feature (`soup`) for the Step 11 TF-IDF similarity engine:
- Multi-word tags, genres, and categories are joined with underscores (e.g. `open_world`, `story_rich`, `single_player`) so they stay as single tokens.
- Repeating a field increases its weight in TF-IDF according to `config.SIM_WEIGHTS = {"genres": 3, "tags": 3, "categories": 1}`.

In [8]:
def join(lst):
    if not isinstance(lst, (list, tuple, set, np.ndarray)):
        return ""
    return " ".join(str(x).lower().replace(" ", "_").replace("-", "_") for x in lst if str(x).strip())

df["genres_str"] = df["genres"].apply(join)
df["tags_str"] = df["tags"].apply(join)
df["categories_str"] = df["categories"].apply(join)

w = config.SIM_WEIGHTS
df["soup"] = (
    (df["genres_str"] + " ") * w["genres"]
    + (df["tags_str"] + " ") * w["tags"]
    + (df["categories_str"] + " ") * w["categories"]
    + df["short_description"].fillna("").str.lower()
).str.replace(r"\s+", " ", regex=True).str.strip()

print("Empty soup count:", (df["soup"] == "").sum())
print("Average soup token count:", df["soup"].str.split().str.len().mean().round(1))

Empty soup count: 48
Average soup token count: 73.8


In [9]:
# Inspect tokenized strings and weighted soup for ELDEN RING
elden = df[df["name"] == "ELDEN RING"].iloc[0]
print(f"=== {elden['name']} ===")
print("genres_str:    ", elden["genres_str"])
print("tags_str:      ", elden["tags_str"])
print("categories_str:", elden["categories_str"])
print("soup preview:  ", elden["soup"][:250] + "...")

=== ELDEN RING ===
genres_str:     action rpg
tags_str:       souls_like dark_fantasy rpg open_world difficult action_rpg third_person multiplayer fantasy action atmospheric singleplayer character_customization co_op story_rich
categories_str: single_player multi_player pvp online_pvp co_op online_co_op steam_achievements full_controller_support steam_trading_cards steam_cloud family_sharing
soup preview:   action rpg action rpg action rpg souls_like dark_fantasy rpg open_world difficult action_rpg third_person multiplayer fantasy action atmospheric singleplayer character_customization co_op story_rich souls_like dark_fantasy rpg open_world difficult a...


## 5.4 Save Deliverable (`games_features.parquet`) and Engineered Features Table

In [10]:
output_path = Path("../data/processed/games_features.parquet")
df.to_parquet(output_path, index=False)

df_check = pd.read_parquet(output_path)
print("Saved data/processed/games_features.parquet successfully!")
print("Verified shape:", df_check.shape)

Saved data/processed/games_features.parquet successfully!
Verified shape: (89550, 65)


In [11]:
engineered_features = pd.DataFrame([
    ["release_year", str(df["release_year"].dtype), "Year extracted from release_date"],
    ["is_free", str(df["is_free"].dtype), "True when price == 0.0"],
    ["price_inr", str(df["price_inr"].dtype), "Price converted to rupees using USD_TO_INR"],
    ["owners_mid", str(df["owners_mid"].dtype), "Midpoint of estimated_owners bracket"],
    ["name_clean", str(df["name_clean"].dtype), "Lowercase, punctuation-stripped game name index"],
    ["n_languages", str(df["n_languages"].dtype), "Count of supported_languages"],
    ["is_singleplayer", str(df["is_singleplayer"].dtype), "Derived from categories ('Single-player')"],
    ["is_multiplayer", str(df["is_multiplayer"].dtype), "Derived from categories ('Multi-player', 'PvP')"],
    ["is_coop", str(df["is_coop"].dtype), "Derived from categories ('Co-op')"],
    ["recency", str(df["recency"].dtype), "Normalized release_year in [0, 1] (newer=high)"],
    ["popularity_score", str(df["popularity_score"].dtype), "log1p(num_reviews_total) blended with owners_mid and peak_ccu"],
    ["rating_score", str(df["rating_score"].dtype), "Bayesian weighted rating (m=165, C=0.7540)"],
    ["genres_str", str(df["genres_str"].dtype), "Underscore-joined lowercase genres tokens"],
    ["tags_str", str(df["tags_str"].dtype), "Underscore-joined lowercase tags tokens"],
    ["categories_str", str(df["categories_str"].dtype), "Underscore-joined lowercase categories tokens"],
    ["soup", str(df["soup"].dtype), "Weighted TF-IDF similarity text (3x genres + 3x tags + 1x categories + summary)"]
], columns=["Feature", "Dtype", "Definition"])

engineered_features.to_csv("../data/processed/engineered_features.csv", index=False)

print("=" * 72)
print("GAMEGUIDE — STEP 5 ENGINEERED FEATURES TABLE")
print("=" * 72)
print(engineered_features.to_string())

GAMEGUIDE — STEP 5 ENGINEERED FEATURES TABLE
            Feature    Dtype                                                                         Definition
0      release_year    int32                                                   Year extracted from release_date
1           is_free     bool                                                             True when price == 0.0
2         price_inr  float64                                       Price converted to rupees using USD_TO_INR
3        owners_mid  float64                                             Midpoint of estimated_owners bracket
4        name_clean   object                                 Lowercase, punctuation-stripped game name index
5       n_languages    int32                                                    Count of supported_languages
6   is_singleplayer     bool                                        Derived from categories ('Single-player')
7    is_multiplayer     bool                                  Derived f

# Step 5 — Feature Engineering Complete

### Completed Tasks
- Verified `release_year`, `is_free`, `price_inr`, and added `name_clean`.
- Engineered `n_languages`, `is_singleplayer`, `is_multiplayer`, `is_coop`, `recency`, and `popularity_score`.
- Computed the **Bayesian weighted rating (`rating_score`)** with $m = 165.0$ and $C = 0.7540$.
- Created underscore-joined `genres_str`, `tags_str`, `categories_str`, and weighted `soup` (`3x genres + 3x tags + 1x categories + short_description`).
- Saved the final engineered dataset as `data/processed/games_features.parquet` (`89,550 rows × 65 columns`).